In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive
import json
from datasets import Dataset
from transformers import Trainer, TrainingArguments

In [2]:
drive.mount('drive')

Mounted at drive


In [3]:
data = json.load(open("/content/drive/MyDrive/Colab/datasets/instruction-data.json"))

In [4]:
data[:3]

[{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
  'input': 'freind --> friend',
  'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'},
 {'instruction': 'Edit the following sentence for grammar.',
  'input': 'He go to the park every day.',
  'output': 'He goes to the park every day.'},
 {'instruction': 'Convert 45 kilometers to meters.',
  'input': '',
  'output': '45 kilometers is 45000 meters.'}]

In [5]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [6]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [7]:
from peft import LoraConfig, get_peft_model

In [8]:
lora_config = LoraConfig(r=50, target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'])
model = get_peft_model(model, lora_config)

In [9]:
model.print_trainable_parameters()

trainable params: 78,643,200 || all params: 3,899,722,752 || trainable%: 2.0166


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [11]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [12]:
len(tokenizer.vocab)

32011

In [13]:
USER_TOKEN = '<|user|>'
ASSISTANT_TOKEN = '<|assistant|>'
SYSTEM_TOKEN = '<|system|>'
END_TOKEN = '<|end|>'

In [14]:
def input_format(example):
  system = 'You are a helpful assistant'
  user = example['instruction'].strip()
  user_in = example['input'].strip()
  assistant = example['output'].strip()
  if user_in:
    user += '\n' + user_in

  text = f'{SYSTEM_TOKEN}\n{system}\n{END_TOKEN}\n'
  text += f'{USER_TOKEN}\n{user}\n{END_TOKEN}\n'
  text += f'{ASSISTANT_TOKEN}\n{assistant}\n{END_TOKEN}'
  return {'text':text}

In [15]:
formatted_data = [input_format(exp) for exp in data]

In [16]:
lens = [len(i['text']) for i in formatted_data]
sum(lens)/len(lens)

189.5981818181818

In [17]:
dataset = Dataset.from_list(formatted_data)

In [18]:
tokenizer.padding_side = 'right'

In [19]:
def prepare_input(text):
  enc = tokenizer(text['text'],
                truncation = True,
                max_length = 96,
                padding = 'max_length'
                )
  input_ids = enc['input_ids']
  labels = [-100]*len(input_ids)
  assistant_id = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
  pad_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
  start = input_ids.index(assistant_id)
  stop = input_ids.index(pad_id) + 1
  for i in range(start, stop):
    labels[i] = input_ids[i]
  return {
      'input_ids': input_ids,
      'attention_mask': enc['attention_mask'],
      'labels': labels
  }

In [20]:
tokenized_ds = dataset.map(prepare_input, remove_columns='text')
tokenized_ds

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1100
})

In [24]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs = 1,
    optim = 'adamw_torch',
    report_to = 'none',
    warmup_steps=200,
    lr_scheduler_type='cosine'
)

In [25]:
next(model.parameters()).device

device(type='cpu')

In [26]:
train = Trainer(
    model=model,
    args = training_args,
    train_dataset=tokenized_ds,
)

In [27]:
next(model.parameters()).device

device(type='cuda', index=0)

In [28]:
train.train()

Step,Training Loss


TrainOutput(global_step=138, training_loss=0.5751421831656194, metrics={'train_runtime': 940.0411, 'train_samples_per_second': 1.17, 'train_steps_per_second': 0.147, 'total_flos': 2408454350438400.0, 'train_loss': 0.5751421831656194, 'epoch': 1.0})

In [29]:
train.save_model('./gphi3_mini_4k_instruct_fine_tune')

In [30]:
!cp -r '/content/gphi3_mini_4k_instruct_fine_tune' '/content/drive/MyDrive/Colab/fine-tune-models'